# Chapter 3: From Biology to Mathematics, Live Python Exercises (Solutions)

**ECUE 3.1.3: Neural Communication** &bull; CESI Bachelor AI (Year 2)

Reference solutions for the Chapter 3 exercises. Try the [exercise notebook](../notebooks/Chapter3_LiveExercises.ipynb) yourself before looking in here.

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Sanity check:</b> run the cell below. If it prints a NumPy version with no error, your environment is ready. If not, go back to <code>Chapter1_Setup.ipynb</code>.
</div>

In [9]:
import numpy as np

print("NumPy version:", np.__version__)
print("Environment OK ✔")

NumPy version: 2.5.4
Environment OK ✔


---

## Part A: Warm-Up, Simulate a Spike Train

*Check your Python and your maths while simulating a neuron.*

We cut time into small bins of $dt = 1$ ms. A neuron firing at $r$ Hz (spikes per second) has, in each bin, a probability

$$p = r \times dt$$

to spike. With 1 ms bins and $r = 80$ Hz, that's $p = 0.08$.

### Step 1: one spike train

Write `spike_train(rate, n_bins, dt, rng)` that returns an array of `n_bins` values, `1` if the neuron spiked in that bin and `0` otherwise.

*Hint*: `rng.random(n)` draws `n` numbers uniformly between 0 and 1. A comparison such as `a < p` gives an array of `True` / `False`, and `.astype(int)` turns it into `1` / `0`.

In [3]:
rng = np.random.default_rng(184)
dt = 0.001        # 1 ms bins
n_bins = 1000     # 1 second

def spike_train(rate, n_bins, dt, rng):
    p = rate * dt                                # spike probability per bin
    return (rng.random(n_bins) < p).astype(int)  # 1 = spike, 0 = silence

### Step 2: weak and strong stimulus

Simulate one second for a **weak** stimulus (5 Hz) and a **strong** one (80 Hz). Count the spikes of each train. **Before running**: what counts do you expect?

In [4]:
weak = spike_train(5, n_bins, dt, rng)
strong = spike_train(80, n_bins, dt, rng)

print("weak  :", weak.sum(), "spikes in 1 s")
print("strong:", strong.sum(), "spikes in 1 s")

weak  : 4 spikes in 1 s
strong: 80 spikes in 1 s


### Solution

The counts are close to 5 and 80 (here 4 and 80), never exactly the same from one run to the next: spiking is random, trial after trial. With another seed, or another order of the random draws, you get other numbers. Dividing a count by the duration (here 1 s) gives the **firing rate** in Hz: that's rate coding.

### Step 3: three neurons at once

Three presynaptic neurons fire at 40, 20 and 10 Hz. Simulate them **in one array** of shape `(3, 1000)`: one row per neuron, one column per bin. Then compute `x`, the vector of their **firing rates** in Hz.

*Hint*: `rates[:, None]` turns a vector of shape `(3,)` into a column of shape `(3, 1)`, so it can be compared with a `(3, 1000)` array. Watch the `axis` of your sum: you want **one rate per neuron**.

In [5]:
rates = np.array([40, 20, 10])

spikes = (rng.random((3, n_bins)) < rates[:, None] * dt).astype(int)
x = spikes.sum(axis=1) / (n_bins * dt)   # count over time / duration = rate in Hz

print("spikes shape:", spikes.shape)
print("firing rates x:", x)

spikes shape: (3, 1000)
firing rates x: [41. 17. 10.]


### Step 4: a weighted sum

Each input has a synaptic weight: $w = (0.5, 0.3, -0.8)$. Compute $\sum_i w_i x_i$ **twice**: first with a `for` loop, then with `np.dot`. You must get the same number.

In [6]:
w = np.array([0.5, 0.3, -0.8])

total = 0.0
for w_i, x_i in zip(w, x):
    total += w_i * x_i

total_dot = np.dot(w, x)

print(total, total_dot)

17.6 17.6


### Solution

Both give the same number (here 17.6), close to $0.5 \times 40 + 0.3 \times 20 - 0.8 \times 10 = 18$: not exactly, because the rates are simulated. `np.dot(w, x)` **is** $\sum_i w_i x_i$. You just went from a simulated biological signal to the input of an artificial neuron: spikes, then counts, then rates, then a weighted sum. That's the pipeline of your projects, in miniature.

### Step 5: counts in time windows

A count over the whole second keeps **how many** spikes, and loses **when** they happened. Cut the second into **4 windows of 250 ms** and count the spikes of each neuron in each window: an array `windows` of shape `(3, 4)`, one row per neuron, one column per window.

*Hint*: `spikes[:, 0:250].sum(axis=1)` counts the first window of every neuron. Loop over the 4 windows. **Before running**: what should each row add up to?

In [7]:
n_windows = 4
width = n_bins // n_windows      # 250 bins = 250 ms

windows = np.zeros((3, n_windows), dtype=int)
for k in range(n_windows):
    windows[:, k] = spikes[:, k * width:(k + 1) * width].sum(axis=1)   # count in window k

print(windows)
print("row sums:", windows.sum(axis=1), "| total counts:", spikes.sum(axis=1))

[[ 9 11  9 12]
 [ 1  3  6  7]
 [ 2  4  1  3]]
row sums: [41 17 10] | total counts: [41 17 10]


### Solution

Each row adds up to the neuron's total count: the windows **split** the count over time, they lose no spike. What they keep is a coarse time axis: one count per window instead of one per trial. It is the PSTH of a single trial, and it gives a longer feature vector (here 4 numbers per neuron instead of 1). Keep this trick in mind for the projects: when two conditions give the same total count, the windows can still tell them apart.

---

## Part B: From $z$ to $y$, in NumPy

*Recap (paper version): a postsynaptic neuron receives 3 synapses.*

| Synapse $i$ | $x_i$ *(Hz)* | $w_i$ |
|:---:|:---:|:---:|
| 1 | 40 | +0.5 |
| 2 | 20 | +0.3 |
| 3 | 10 | &minus;0.8 |

**Bias:** $b = -25$

Recall the full equation:

$$z = \sum_{i=1}^{n} w_i x_i + b \qquad\qquad y = \varphi(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

**Before running anything: what did you get by hand for $z$ and $y$?** Write your prediction down, then check it below.

In [8]:
x = np.array([40, 20, 10])
w = np.array([0.5, 0.3, -0.8])
b = -25

z = np.dot(w, x) + b        # weighted sum + bias
y = 1 if z >= 0 else 0      # step function

print("z =", z)
print("y =", y)

z = -7.0
y = 0


### Solution

Notice the mapping: `np.dot(w, x)` **is** $\sum_i w_i x_i$, `+ b` **is** the bias, and the one-line conditional **is** the step function. Nothing new was invented: the same formula, just executed by a machine instead of by hand.

---

## Chapter Bilan: Rebuild a Neuron From Scratch

*Synthesis exercise. No scaffolding this time: given a brand new synapse matrix, write the **entire artificial neuron** yourself, from raw inputs to the final binary decision.*

**A different postsynaptic neuron, now with 5 synapses:**

| Synapse $i$ | $x_i$ *(Hz)* | $w_i$ |
|:---:|:---:|:---:|
| 1 | 60 | +0.4 |
| 2 | 15 | +0.2 |
| 3 | 25 | &minus;0.6 |
| 4 | 5  | +0.9 |
| 5 | 45 | &minus;0.3 |

**Bias:** $b = -18$

### Your turn

Write a single function `neuron(x, w, b)` implementing $y = \varphi\left(\sum_i w_i x_i + b\right)$ from scratch, then apply it to the matrix above and report both $z$ and $y$. Predict them by hand first.

In [10]:
def neuron(x, w, b):
    z = np.dot(w, x) + b       # weighted sum + bias (spatial summation)
    y = 1 if z >= 0 else 0     # step function (all-or-nothing decision)
    return y

x = np.array([60, 15, 25, 5, 45])
w = np.array([0.4, 0.2, -0.6, 0.9, -0.3])
b = -18

print("w_1 . x_1 =", 60 * 0.4)
print("w_2 . x_2 =", 15 * 0.2)
print("w_3 . x_3 =", 25 * -0.6)
print("w_4 . x_4 =", 5 * 0.9)
print("w_5 . x_5 =", 45 * -0.3)

print("z =", np.dot(w, x) + b)
print("y =", neuron(x, w, b))

w_1 . x_1 = 24.0
w_2 . x_2 = 3.0
w_3 . x_3 = -15.0
w_4 . x_4 = 4.5
w_5 . x_5 = -13.5
z = -15.0
y = 0


### Solution

$z = -15$, so $y = 0$: three excitatory contributions (24, 3 and 4.5) are pulled down by two inhibitory ones (-15 and -13.5), and the bias of -18 finishes the job. Integration is not a majority vote, it's a weighted sum.

This `neuron` function is not a toy: it already **is** the McCulloch-Pitts neuron we name and generalize in Chapter 4.